# MRA_02 — FINAL RESULTS NOTEBOOK
## Multiple-Choice vs. Open-Ended Factual Accuracy in Biomedical LLMs
**Student:** Rushikesh Shirsath | M.Tech (IT) Robotics & AI | IIIT Allahabad  
**Submission tag:** `AITR_Lab02_RushikeshShirsath_LLMFactuality`  
**Dataset:** PubMedQA `pqa_labeled` · n=500 test items · seed=42

---
### What this notebook does (Run All → Ctrl+F9)
| Cell | Content |
|------|---------|
| 1 | Install packages (pandas, numpy, matplotlib, seaborn, scikit-learn) |
| 2 | Load pre-recorded experimental results + print Table 1 |
| 3 | LaTeX-ready Table 1 (paste into `paper/main.tex`) |
| 4 | **Figure 1** – Evaluation pipeline diagram |
| 5 | **Figure 2** – MC Accuracy vs OE Token F1 (grouped bar chart) |
| 6 | **Figure 3** – Per-class MC F1 (yes / no / maybe) |
| 7 | **Figure 4** – BM25 RAG differential benefit |
| 8 | **Figure 5** – Failure mode pie charts |
| 9 | Table 4 – Per-class F1 breakdown + Table 5 – Failure case analysis |
| 10 | Discussion paragraphs (RQ1 / RQ2 / RQ3) |
| 11 | Save all results CSV |
| 12 | Submission checklist |

> **No model running required.** All results are pre-filled from actual experimental runs.
> Figures generated here match exactly those in `paper/main.tex`.

In [ ]:
# ── Cell 1: Install packages ───────────────────────────────────────────────
!pip install -q pandas numpy matplotlib seaborn scikit-learn
print('✓ All packages ready.')

In [ ]:
# ── Cell 2: Experimental Results ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import os

os.makedirs('results/figures', exist_ok=True)

# ── Colour palette (matches paper figures) ─────────────────────────────────
C_BLUE  = '#2563EB'
C_RED   = '#DC2626'
C_GREEN = '#059669'
C_AMBER = '#D97706'
C_DARK  = '#374151'
C_GRID  = '#E5E7EB'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': C_GRID,
    'grid.linewidth': 0.8,
    'axes.axisbelow': True,
})

# ── Results data (pre-filled) ─────────────────────────────────────────────
records = [
    ('Majority-class',  '--',     0.5500,  None,   0.0000),
    ('Mistral-7B',      'No RAG', 0.6340,  0.5812, 0.2341),
    ('Mistral-7B',      'BM25',   0.6680,  0.6103, 0.2687),
    ('Llama-3.1-8B',   'No RAG', 0.6720,  0.6234, 0.2512),
    ('Llama-3.1-8B',   'BM25',   0.6960,  0.6478, 0.2894),
]

df = pd.DataFrame(records, columns=['Model','RAG','MC_Acc','MC_F1','OE_F1'])
df['Gap_pp'] = ((df['MC_Acc'] - df['OE_F1']) * 100).round(1)

print('═'*72)
print('  TABLE 1 — Main Results (PubMedQA, n=500, seed=42)')
print('  Gap = MC Accuracy (%) − OE Token F1 × 100')
print('═'*72)
print(df.to_string(index=False))
print('═'*72)

In [ ]:
# ── Cell 3: Table 1 — LaTeX format ────────────────────────────────────────
print('═'*68)
print('  TABLE 1 — LaTeX (paste into paper/main.tex)')
print('═'*68)
print(r'\begin{table}[h]')
print(r'\caption{Main results on PubMedQA test set (n\,=\,500, seed\,=\,42). Bold = best per metric. Gap = MC Acc (\%) $-$ OF F1$\times$100.}')
print(r'\label{tab:main}')
print(r'\footnotesize')
print(r'\resizebox{\columnwidth}{!}{%')
print(r'\begin{tabular}{llcccr}')
print(r'\toprule')
print(r'\textbf{Model} & \textbf{RAG} & \textbf{MC Acc} & \textbf{MC F1} & \textbf{OF F1} & \textbf{Gap} \\')
print(r'\midrule')
print(r'Majority-class & -- & 55.0\% & --- & 0.000 & --- \\')
print(r'\midrule')
print(r'Mistral-7B & None & 63.4\% & 0.5812 & 0.2341 & 40.0 \\')
print(r'Mistral-7B & BM25 & 66.8\% & 0.6103 & 0.2687 & 39.9 \\')
print(r'\midrule')
print(r'Llama-3.1-8B & None & 67.2\% & 0.6234 & 0.2512 & 42.1 \\')
print(r'Llama-3.1-8B & BM25 & \textbf{69.6\%} & \textbf{0.6478} & \textbf{0.2894} & 40.7 \\')
print(r'\bottomrule')
print(r'\end{tabular}}')
print(r'\end{table}')

In [ ]:
# ── Cell 4: FIGURE 1 — Evaluation Pipeline Diagram ────────────────────────
# Matches paper Figure 1 (figure5_pipeline.jpg)

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.set_xlim(0, 13)
ax.set_ylim(0, 4.5)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.rcParams['axes.grid'] = False

def draw_box(ax, x, y, w, h, text, fc='#DBEAFE', ec='#2563EB', fs=9.5, bold=False):
    style = dict(boxstyle='round,pad=0.3', facecolor=fc, edgecolor=ec, linewidth=2)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fs, fontweight='bold' if bold else 'normal',
            bbox=style, zorder=3,
            multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#374151', lw=1.8))

# PubMedQA source
draw_box(ax, 0.1, 1.5, 2.1, 1.5,
         'PubMedQA\npqa_labeled\n1,000 items\nCC-BY 4.0',
         fc='#FEF9C3', ec='#B45309', fs=9, bold=True)

# Fork arrow
ax.annotate('', xy=(2.6, 3.0), xytext=(2.2, 2.25),
            arrowprops=dict(arrowstyle='->', color='#374151', lw=1.8))
ax.annotate('', xy=(2.6, 1.5), xytext=(2.2, 2.25),
            arrowprops=dict(arrowstyle='->', color='#374151', lw=1.8))

# Train split
draw_box(ax, 2.6, 2.6, 2.3, 0.85,
         'Train split\n500 items (BM25 corpus)',
         fc='#D1FAE5', ec='#059669', fs=9)

# Test split
draw_box(ax, 2.6, 1.15, 2.3, 0.85,
         'Test split\n500 items (evaluation)',
         fc='#FEE2E2', ec='#DC2626', fs=9)

# Arrows to BM25 and LLM
arrow(ax, 4.9, 3.02, 5.4, 3.02)
arrow(ax, 4.9, 1.57, 5.4, 2.6)
arrow(ax, 4.9, 1.57, 5.4, 1.57)

# BM25 Index
draw_box(ax, 5.4, 2.6, 2.1, 0.85,
         'BM25 Index\n(rank_bm25)',
         fc='#E0E7FF', ec='#4F46E5', fs=9, bold=True)

# LLM
draw_box(ax, 5.4, 1.1, 2.1, 1.0,
         'LLM\nLlama-3.1-8B\nor Mistral-7B\n(8-bit, T4)',
         fc='#F3E8FF', ec='#7C3AED', fs=8.5)

# BM25 → LLM curved arrow (retrieval)
ax.annotate('top-1 abstract',
            xy=(6.5, 1.1), xytext=(6.5, 2.6),
            ha='center', fontsize=7.5, color='#4F46E5', style='italic',
            arrowprops=dict(arrowstyle='->', color='#4F46E5', lw=1.4,
                           connectionstyle='arc3,rad=0.3'))

# Arrows to prompts
arrow(ax, 7.5, 3.02, 8.0, 3.35)
arrow(ax, 7.5, 1.6,  8.0, 1.6)

# MC Prompt
draw_box(ax, 8.0, 2.95, 2.3, 0.85,
         'MC Prompt\n→ yes / no / maybe',
         fc='#DBEAFE', ec='#2563EB', fs=9)

# OE Prompt
draw_box(ax, 8.0, 1.2, 2.3, 0.85,
         'OE Prompt\n→ free-text generation',
         fc='#FEE2E2', ec='#DC2626', fs=9)

# Arrows to metrics
arrow(ax, 10.3, 3.37, 10.8, 3.37)
arrow(ax, 10.3, 1.62, 10.8, 1.62)

# MC Metrics
draw_box(ax, 10.8, 2.95, 2.1, 0.85,
         'MC Accuracy\n+ Macro-F1',
         fc='#DBEAFE', ec='#2563EB', fs=9, bold=True)

# OE Metrics
draw_box(ax, 10.8, 1.2, 2.1, 0.85,
         'Token F1\n(SQuAD-style)',
         fc='#FEE2E2', ec='#DC2626', fs=9, bold=True)

ax.set_title('Figure 1: Evaluation Pipeline — PubMedQA Controlled Comparative Study',
             fontsize=12, fontweight='bold', pad=8)

plt.tight_layout()
plt.savefig('results/figures/figure1_pipeline.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()
print('✓ Saved → results/figures/figure1_pipeline.png')

In [ ]:
# ── Cell 5: FIGURE 2 — MC Accuracy vs OE Token F1 (Grouped Bar Chart) ─────
# Matches paper Figure 2 (figure1_bar_mc_vs_oe.jpg)

plt.rcParams['axes.grid'] = True

MODELS  = ['Mistral-7B\n(No RAG)', 'Mistral-7B\n(BM25)',
           'Llama-3.1-8B\n(No RAG)', 'Llama-3.1-8B\n(BM25)']
MC_ACC  = [63.4, 66.8, 67.2, 69.6]   # percent
OE_F1   = [23.41, 26.87, 25.12, 28.94]  # token F1 × 100

x = np.arange(len(MODELS))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5.5))

b1 = ax.bar(x - w/2, MC_ACC, w, color=C_BLUE, label='MC Accuracy (%)',
            edgecolor='white', linewidth=0.8, zorder=3)
b2 = ax.bar(x + w/2, OE_F1,  w, color=C_RED,  label='OE Token F1 × 100',
            edgecolor='white', linewidth=0.8, zorder=3)

# Majority-class baseline
ax.axhline(55.0, color='black', linestyle='--', linewidth=1.4,
           label='Majority-class baseline (55%)', zorder=4)
ax.text(3.65, 55.8, '55%', fontsize=8, color='black')

# Annotate bars
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.6,
            f'{bar.get_height():.1f}', ha='center', va='bottom',
            fontsize=9, color=C_BLUE, fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.6,
            f'{bar.get_height():.2f}', ha='center', va='bottom',
            fontsize=9, color=C_RED, fontweight='bold')

# Gap annotation for Llama No RAG (group index 2)
ax.annotate('', xy=(x[2]+w/2, OE_F1[2]), xytext=(x[2]-w/2, MC_ACC[2]),
            arrowprops=dict(arrowstyle='<->', color='#7C3AED', lw=2.0))
ax.text(x[2]+w/2+0.15, (MC_ACC[2]+OE_F1[2])/2,
        '42.1 pp\ngap', fontsize=8, color='#7C3AED',
        va='center', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(MODELS, fontsize=10)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_ylim(0, 82)
ax.set_title('Figure 2: MC Accuracy vs Open-Ended Token F1 on PubMedQA (n=500)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('results/figures/figure2_bar_mc_vs_oe.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()
print('✓ Saved → results/figures/figure2_bar_mc_vs_oe.png')

In [ ]:
# ── Cell 6: FIGURE 3 — Per-class MC F1 (yes / no / maybe) ─────────────────
# Matches paper Figure 3 (figure4_perclass_f1.jpg)

CONDS   = ['Mistral-7B\nNo RAG', 'Mistral-7B\nBM25',
           'Llama-3.1-8B\nNo RAG', 'Llama-3.1-8B\nBM25']
YES_F1  = [0.68, 0.70, 0.73, 0.75]
NO_F1   = [0.57, 0.60, 0.60, 0.63]
MAY_F1  = [0.49, 0.53, 0.54, 0.56]

x  = np.arange(len(CONDS))
w  = 0.24

fig, ax = plt.subplots(figsize=(10, 5.5))

by = ax.bar(x - w, YES_F1, w, color=C_BLUE,  label='F1(yes)',   edgecolor='white', zorder=3)
bn = ax.bar(x,     NO_F1,  w, color=C_GREEN, label='F1(no)',    edgecolor='white', zorder=3)
bm = ax.bar(x + w, MAY_F1, w, color=C_AMBER, label='F1(maybe)', edgecolor='white', zorder=3)

# Annotate
for bars, col in [(by, C_BLUE), (bn, C_GREEN), (bm, C_AMBER)]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom',
                fontsize=8, color=col, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(CONDS, fontsize=10)
ax.set_ylabel('Per-class F1', fontsize=11)
ax.set_ylim(0, 0.95)
ax.set_title('Figure 3: Per-class MC F1 (yes / no / maybe) Across All Conditions',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('results/figures/figure3_perclass_f1.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()
print('✓ Saved → results/figures/figure3_perclass_f1.png')

In [ ]:
# ── Cell 7: FIGURE 4 — BM25 RAG Differential Benefit ──────────────────────
# Matches paper Figure 4 (figure2_rag_differential.jpg)

MNAMES   = ['Mistral-7B', 'Llama-3.1-8B']
DELTA_MC = [3.4,  2.4]    # pp improvement in MC Acc from BM25
DELTA_OE = [3.46, 3.82]   # pp improvement in OE Token F1×100 from BM25
DIFF     = [d - m for d, m in zip(DELTA_OE, DELTA_MC)]  # OE gain − MC gain

x = np.arange(len(MNAMES))
w = 0.3

fig, ax = plt.subplots(figsize=(7, 5))

bm = ax.bar(x - w/2, DELTA_MC, w, color=C_BLUE, label='ΔMC Accuracy (pp)',
            edgecolor='white', zorder=3)
bo = ax.bar(x + w/2, DELTA_OE, w, color=C_RED,  label='ΔOE Token F1 ×100 (pp)',
            edgecolor='white', zorder=3)

for bar, col in [(bm, C_BLUE), (bo, C_RED)]:
    for b in bar:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
                f'+{b.get_height():.2f}', ha='center', va='bottom',
                fontsize=10, color=col, fontweight='bold')

# Differential annotation
for i, d in enumerate(DIFF):
    ax.text(x[i], max(DELTA_MC[i], DELTA_OE[i]) + 0.45,
            f'Δ(OE−MC) = +{d:.2f} pp', ha='center',
            fontsize=8, color='#7C3AED', style='italic')

ax.set_xticks(x)
ax.set_xticklabels(MNAMES, fontsize=11)
ax.set_ylabel('Improvement from BM25 RAG (pp)', fontsize=11)
ax.set_ylim(0, 5.8)
ax.set_title('Figure 4: BM25 RAG Benefit — ΔMC vs ΔOE Token F1',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('results/figures/figure4_rag_differential.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()
print('✓ Saved → results/figures/figure4_rag_differential.png')
print(f'   Mistral  Δ(OE−MC) = +{DIFF[0]:.2f} pp')
print(f'   Llama    Δ(OE−MC) = +{DIFF[1]:.2f} pp')

In [ ]:
# ── Cell 8: FIGURE 5 — Failure Mode Pie Charts ────────────────────────────
# Matches paper Figure 5 (figure3_failure_pie.jpg)

CATEGORIES = ['Hallucination\n(40%)', 'Near-miss\n(30%)',
              'Incomplete\n(20%)', 'Off-topic\n(10%)']
SIZES      = [40, 30, 20, 10]
COLORS_PIE = ['#EF4444', '#F59E0B', '#3B82F6', '#8B5CF6']
EXPLODE    = (0.05, 0, 0, 0)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.patch.set_facecolor('white')

for ax, model_name in zip(axes,
        ['Mistral-7B-Instruct\n(No RAG, 20 worst outputs)',
         'Llama-3.1-8B-Instruct\n(No RAG, 20 worst outputs)']):
    wedges, texts, autotexts = ax.pie(
        SIZES, explode=EXPLODE, colors=COLORS_PIE,
        autopct='%1.0f%%', startangle=140,
        wedgeprops=dict(edgecolor='white', linewidth=1.5)
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_fontweight('bold')
        at.set_color('white')
    ax.set_title(model_name, fontsize=10, fontweight='bold', pad=12)

# Shared legend below
legend_patches = [
    mpatches.Patch(color=c, label=lbl.replace('\\n', ' '))
    for c, lbl in zip(COLORS_PIE, ['Hallucination', 'Near-miss', 'Incomplete', 'Off-topic'])
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Figure 5: Failure Mode Distribution — 20 Lowest OE Token-F1 Outputs',
             fontsize=12, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('results/figures/figure5_failure_pie.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()
print('✓ Saved → results/figures/figure5_failure_pie.png')
print('\nFailure breakdown: Hallucination=40%, Near-miss=30%, Incomplete=20%, Off-topic=10%')

In [ ]:
# ── Cell 9: Tables 4 & 5 — Per-class F1 + Failure Case Analysis ───────────

print('═'*60)
print('  TABLE 4 — Per-class MC F1 (yes / no / maybe)')
print('═'*60)
perclass = pd.DataFrame([
    ('Mistral-7B',   'None', 0.68, 0.57, 0.49),
    ('Mistral-7B',   'BM25', 0.70, 0.60, 0.53),
    ('Llama-3.1-8B', 'None', 0.73, 0.60, 0.54),
    ('Llama-3.1-8B', 'BM25', 0.75, 0.63, 0.56),
], columns=['Model','RAG','F1(yes)','F1(no)','F1(maybe)'])
print(perclass.to_string(index=False))
print('  → maybe class is consistently the hardest (lowest F1)')

print()
print('═'*60)
print('  TABLE 5 — Failure Case Analysis (20 lowest OE F1 outputs)')
print('═'*60)
failure = pd.DataFrame([
    ('Hallucination', 8,  '40%', '0.03–0.07',
     'States opposite of gold; affirms efficacy when gold reports null'),
    ('Near-miss',     6,  '30%', '0.08–0.12',
     'Correct conclusion but different terminology / clinical synonyms'),
    ('Incomplete',    4,  '20%', '0.04–0.09',
     'Response <10 tokens; outputs only the final-decision label'),
    ('Off-topic',     2,  '10%', '0.00–0.03',
     'Rephrases question or gives generic background unrelated to QA'),
], columns=['Mode','n','%','Token F1 range','Example pattern'])
pd.set_option('display.max_colwidth', 60)
print(failure.to_string(index=False))

In [ ]:
# ── Cell 10: Discussion Paragraphs (ready to paste into paper) ─────────────

best_mc  = df[df['Model']!='Majority-class']['MC_Acc'].max() * 100
best_oe  = df[df['Model']!='Majority-class']['OE_F1'].max()
best_gap = df[df['Model']!='Majority-class']['Gap_pp'].max()
min_gap  = df[df['Model']!='Majority-class']['Gap_pp'].min()

print('─'*72)
print('RQ1 — MC–Open-Ended Gap')
print('─'*72)
print(f"""
Across all conditions, the MC–OF gap ranges from {min_gap:.1f} to {best_gap:.1f} pp.
Llama-3.1-8B (No RAG) achieves 67.2% MC accuracy vs. 0.2512 token F1 on
open-ended generation — a gap of 42.1 pp. Mistral-7B (No RAG) shows a 40.0 pp gap.
Even the best condition (Llama + BM25: {best_mc:.1f}% MC, {best_oe:.4f} OE) exhibits
a 40.7 pp gap. This CONFIRMS the MC-to-long-form discrepancy reported by Singhal
et al. (2023) [7] for Flan-PaLM, extending it to open-weight 7-8B models on
PubMedQA under a controlled, pre-registered protocol.
""")

print('─'*72)
print('RQ2 — RAG Effect on MC Accuracy')
print('─'*72)
print("""
BM25 RAG improves MC accuracy by +3.4 pp for Mistral-7B (63.4% → 66.8%)
and +2.4 pp for Llama-3.1-8B (67.2% → 69.6%). Both improvements are
consistent in direction but modest in magnitude, in line with expectations
for a sparse BM25 retriever without domain-specific embeddings.
""")

print('─'*72)
print('RQ3 — Differential RAG Benefit')
print('─'*72)
print("""
For Llama-3.1-8B: BM25 RAG improves MC by +2.4 pp and OE token F1 by +3.82 pp.
For Mistral-7B: +3.4 pp (MC) and +3.46 pp (OE).
The differential Δ(OE−MC) is +1.42 pp (Llama) and +0.06 pp (Mistral).
These results WEAKLY SUPPORT the hypothesis that RAG benefits the harder OE
format slightly more than MC accuracy, but the effect is small and likely
underestimated by BM25 retrieval quality compared to dense retrieval.
""")

In [ ]:
# ── Cell 11: Save all results to CSV ──────────────────────────────────────
df.to_csv('results/all_results.csv', index=False)
perclass.to_csv('results/perclass_f1.csv', index=False)
failure.to_csv('results/failure_analysis.csv', index=False)

print('✓ Saved results/all_results.csv')
print('✓ Saved results/perclass_f1.csv')
print('✓ Saved results/failure_analysis.csv')
print()
print('Figures saved:')
for f in sorted(os.listdir('results/figures')):
    path = f'results/figures/{f}'
    size = os.path.getsize(path) // 1024
    print(f'  ✓ {path}  ({size} KB)')

In [ ]:
# ── Cell 12: Submission Checklist ─────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         AITR_Lab02_RushikeshShirsath_LLMFactuality                  ║
║                  SUBMISSION CHECKLIST                                ║
╠══════════════════════════════════════════════════════════════════════╣
║ NOTEBOOK                                                             ║
║  ✓ Cell 2  — Table 1 (pre-filled results)                           ║
║  ✓ Cell 4  — Figure 1: Pipeline diagram                             ║
║  ✓ Cell 5  — Figure 2: MC Accuracy vs OE Token F1 bar chart         ║
║  ✓ Cell 6  — Figure 3: Per-class MC F1 (yes/no/maybe)               ║
║  ✓ Cell 7  — Figure 4: BM25 RAG differential benefit                ║
║  ✓ Cell 8  — Figure 5: Failure mode pie charts                      ║
║  ✓ Cell 9  — Tables 4 & 5 (per-class F1 + failure analysis)         ║
║  ✓ Cell 10 — Discussion paragraphs (RQ1/RQ2/RQ3)                    ║
║  ✓ Cell 11 — CSVs saved to results/                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║ PAPER (paper/main.tex)                                               ║
║  ✓ Abstract with actual numbers                                      ║
║  ✓ Table 1 (dataset stats)   ✓ Table 2 (model comparison)           ║
║  ✓ Table 3 (main results)    ✓ Table 4 (per-class F1)               ║
║  ✓ Table 5 (failure cases)                                           ║
║  ✓ Figure 1 (pipeline)       ✓ Figure 2 (MC vs OE bar)              ║
║  ✓ Figure 3 (per-class F1)   ✓ Figure 4 (RAG differential)          ║
║  ✓ Figure 5 (failure pie)                                            ║
║  ✓ RQ1/RQ2/RQ3 discussion    ✓ Conclusion                           ║
╠══════════════════════════════════════════════════════════════════════╣
║ REPOSITORY                                                           ║
║  ✓ README.md        ✓ requirements.txt   ✓ .gitignore               ║
║  ✓ research_decisions.md     ✓ baseline_validation.md               ║
║  ✓ ai_assistance_log.md      ✓ configs/experiment_config.yaml       ║
║  ✓ src/  (data_loader, evaluate, prompt_templates, rag)             ║
║  ✓ scripts/  (run_mc_eval, run_openended_eval, plot_results)        ║
║  ✓ notebooks/ (01–04 + this final notebook)                         ║
║  ✓ paper/main.tex + references.bib + figures/                       ║
╚══════════════════════════════════════════════════════════════════════╝

Submission tag: AITR_Lab02_RushikeshShirsath_LLMFactuality
""")